# 01 — Data Prep + Gemini Sentiment Scoring
### E-commerce Customer Churn Prediction — ML Pipeline (Part 1/2)

**Pipeline stage covered here:** `Customer Data -> Gemini Sentiment -> (merged dataset ready for training)`

This notebook:
1. Downloads two Kaggle datasets:
   - **Ecommerce Customer Churn Analysis and Prediction** (ankitverma2010) — structured customer behaviour features + ready-made `Churn` label (Tenure, OrderCount, DaySinceLastOrder, Complain, CashbackAmount, SatisfactionScore, ...).
   - **Women's E-Commerce Clothing Reviews** (nicapotato) — real English review text + star rating.
2. Since no single public dataset joins churn behaviour + review text for the *same* customer, we **simulate a realistic link**: each churn customer is assigned a review whose sentiment tier is correlated with that customer's `Complain` / `SatisfactionScore` fields (unhappy customers get sampled mostly from low-rating reviews, happy customers mostly from high-rating reviews, with noise mixed in so it isn't a trivial leak).
3. Scores every assigned review with the **Gemini API** to get a `sentiment_score` (-1..1) + `sentiment_label`.
4. Saves the merged, sentiment-enriched dataset to Google Drive as `churn_with_sentiment.csv` — this is the input for Notebook 2 (model training).

> Run cells top-to-bottom. Re-running this notebook re-spends Gemini API quota, so once `churn_with_sentiment.csv` is saved you normally only need Notebook 2 again.

## 1. Setup

In [ ]:
!pip install -q kaggle google-genai pandas openpyxl tqdm

## 2. Kaggle API credentials

1. Go to https://www.kaggle.com/settings -> **API** -> **Create New Token**. This downloads `kaggle.json`.
2. Run the cell below and upload that file when prompted.

In [ ]:
import os
from google.colab import files

os.makedirs("/root/.kaggle", exist_ok=True)
uploaded = files.upload()  # select your kaggle.json here

for fname in uploaded:
    if fname != "kaggle.json":
        continue
    with open("/root/.kaggle/kaggle.json", "wb") as f:
        f.write(uploaded[fname])
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("Kaggle credentials installed.")

## 3. Download datasets

In [ ]:
!kaggle datasets download -d ankitverma2010/ecommerce-customer-churn-analysis-and-prediction -p /content/data --unzip
!kaggle datasets download -d nicapotato/womens-ecommerce-clothing-reviews -p /content/data --unzip
!ls -la /content/data

## 4. Load the churn dataset

The churn dataset ships as an Excel workbook (`E Commerce Dataset.xlsx`) with the data in sheet `E Comm`. Column names can vary slightly by upload, so we normalise them defensively.

In [ ]:
import pandas as pd
import glob

xlsx_path = glob.glob("/content/data/*.xlsx")[0]
churn_df = pd.read_excel(xlsx_path, sheet_name="E Comm")
churn_df.columns = [c.strip() for c in churn_df.columns]

print(churn_df.shape)
churn_df.head()

In [ ]:
# Quick sanity check on expected columns
expected_cols = [
    "CustomerID", "Churn", "Tenure", "PreferredLoginDevice", "CityTier",
    "WarehouseToHome", "PreferredPaymentMode", "Gender", "HourSpendOnApp",
    "NumberOfDeviceRegistered", "PreferedOrderCat", "SatisfactionScore",
    "MaritalStatus", "NumberOfAddress", "Complain",
    "OrderAmountHikeFromlastYear", "CouponUsed", "OrderCount",
    "DaySinceLastOrder", "CashbackAmount",
]
missing = [c for c in expected_cols if c not in churn_df.columns]
print("Missing columns (investigate if non-empty):", missing)

churn_df.isna().sum().sort_values(ascending=False)

In [ ]:
# Basic cleaning: numeric NaNs -> median (these are small counts of missing behavioural fields)
numeric_cols = [
    "Tenure", "WarehouseToHome", "HourSpendOnApp",
    "OrderAmountHikeFromlastYear", "CouponUsed", "OrderCount", "DaySinceLastOrder",
]
for c in numeric_cols:
    if c in churn_df.columns:
        churn_df[c] = churn_df[c].fillna(churn_df[c].median())

churn_df = churn_df.drop_duplicates(subset="CustomerID").reset_index(drop=True)
print(churn_df.shape)
churn_df["Churn"].value_counts(normalize=True)

## 5. Load the reviews dataset and bucket by rating

In [ ]:
reviews_path = "/content/data/Womens Clothing E-Commerce Reviews.csv"
reviews_df = pd.read_csv(reviews_path, index_col=0)
reviews_df = reviews_df.dropna(subset=["Review Text"]).reset_index(drop=True)
reviews_df["Review Text"] = reviews_df["Review Text"].str.strip()
reviews_df = reviews_df[reviews_df["Review Text"].str.len() > 15].reset_index(drop=True)

print(reviews_df.shape)
reviews_df[["Rating", "Recommended IND", "Review Text"]].head()

In [ ]:
# Bucket reviews into 3 tiers by star rating: negative (1-2), neutral (3), positive (4-5)
def rating_tier(r):
    if r <= 2:
        return "negative"
    if r == 3:
        return "neutral"
    return "positive"

reviews_df["tier"] = reviews_df["Rating"].apply(rating_tier)
reviews_by_tier = {t: g.reset_index(drop=True) for t, g in reviews_df.groupby("tier")}
{t: len(g) for t, g in reviews_by_tier.items()}

## 6. Simulate the customer <-> review link

Each customer's underlying "happiness" tier is derived from `Complain` and `SatisfactionScore`:
- `Complain == 1` or `SatisfactionScore <= 2` -> mostly **negative** reviews
- `SatisfactionScore == 3` -> mostly **neutral**
- `SatisfactionScore >= 4` and no complaint -> mostly **positive**

We assign the *dominant* tier ~70% of the time and mix in the other tiers ~30% of the time, so the sentiment signal correlates with churn behaviour but isn't a perfect leak (matches how noisy real review sentiment actually is).

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

def customer_tier(row):
    if row["Complain"] == 1 or row["SatisfactionScore"] <= 2:
        return "negative"
    if row["SatisfactionScore"] == 3:
        return "neutral"
    return "positive"

tier_mix = {
    "negative": {"negative": 0.70, "neutral": 0.20, "positive": 0.10},
    "neutral":  {"negative": 0.20, "neutral": 0.60, "positive": 0.20},
    "positive": {"negative": 0.10, "neutral": 0.20, "positive": 0.70},
}

def sample_review_text(dominant_tier):
    tiers = list(tier_mix[dominant_tier].keys())
    probs = list(tier_mix[dominant_tier].values())
    chosen_tier = rng.choice(tiers, p=probs)
    pool = reviews_by_tier[chosen_tier]
    row = pool.iloc[rng.integers(0, len(pool))]
    return row["Review Text"], row["Rating"]

churn_df["dominant_tier"] = churn_df.apply(customer_tier, axis=1)

review_texts, review_ratings = [], []
for tier in churn_df["dominant_tier"]:
    text, rating = sample_review_text(tier)
    review_texts.append(text)
    review_ratings.append(rating)

churn_df["review_text"] = review_texts
churn_df["review_rating"] = review_ratings

churn_df[["CustomerID", "Complain", "SatisfactionScore", "dominant_tier", "review_rating", "review_text"]].head(10)

## 7. Gemini sentiment scoring

Set your Gemini API key as a Colab secret (key icon in the left sidebar -> add secret named `GEMINI_API_KEY`), or paste it via `getpass` below. **Never hardcode the key in the notebook.**

Uses the current `google-genai` SDK (the older `google.generativeai` package is being retired and some models under it now 404). Retries handle transient `503`/`429` errors from Gemini with exponential backoff.

In [ ]:
from google import genai
from google.genai import types

try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except Exception:
    api_key = None

if not api_key:
    from getpass import getpass
    api_key = getpass("Enter your Gemini API key: ")

client = genai.Client(api_key=api_key)
MODEL_NAME = "gemini-3.5-flash-lite"
print("Gemini client ready.")

In [ ]:
import json
import time
import random
from tqdm import tqdm

SENTIMENT_PROMPT = """
You are a sentiment analysis engine for e-commerce customer reviews.

Read the review below and return ONLY a compact JSON object.

{{
  "sentiment_score": <float from -1.0 to 1.0>,
  "sentiment_label": "<negative|neutral|positive>"
}}

Rules:
- -1.0 = very negative
- 0.0 = neutral
- 1.0 = very positive
- Return ONLY JSON
- No markdown
- No explanation

Review:
\"\"\"{review}\"\"\"
"""

def get_sentiment(review_text, retries=5):
    if review_text is None:
        return 0.0, "neutral"

    review_text = str(review_text).strip()
    if not review_text:
        return 0.0, "neutral"

    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=SENTIMENT_PROMPT.format(review=review_text[:2000]),
                config=types.GenerateContentConfig(
                    response_mime_type="application/json"
                ),
            )

            data = json.loads(response.text)
            score = float(data["sentiment_score"])
            label = str(data["sentiment_label"]).lower().strip()

            score = max(-1.0, min(1.0, score))
            if label not in ["negative", "neutral", "positive"]:
                label = "neutral"

            return score, label

        except Exception as e:
            error = str(e)

            if "503" in error or "UNAVAILABLE" in error:
                wait_time = min(60, (2 ** attempt) + random.uniform(0, 2))
                print(f"\\nGemini temporarily unavailable. Retry {attempt + 1}/{retries} after {wait_time:.1f}s...")
                time.sleep(wait_time)
                continue

            if "429" in error or "RESOURCE_EXHAUSTED" in error:
                wait_time = min(120, 5 * (2 ** attempt))
                print(f"\\nRate limit reached. Waiting {wait_time}s...")
                time.sleep(wait_time)
                continue

            if attempt == retries - 1:
                print("\\nSentiment call failed:", error)
                return 0.0, "neutral"

            time.sleep(2)

    return 0.0, "neutral"

# NOTE: this calls the Gemini API once per row. For the free tier, consider
# slicing churn_df (e.g. churn_df.head(500)) first if you hit rate limits,
# then resume with the remaining rows in a second pass.
scores, labels = [], []
for text in tqdm(churn_df["review_text"], desc="Scoring sentiment"):
    s, l = get_sentiment(text)
    scores.append(s)
    labels.append(l)
    time.sleep(1)  # gentle pacing to avoid rate-limit errors

churn_df["sentiment_score"] = scores
churn_df["sentiment_label"] = labels
churn_df[["CustomerID", "review_rating", "sentiment_score", "sentiment_label"]].head(10)

In [ ]:
# Sanity check: sentiment_score should correlate with review_rating
churn_df.groupby("review_rating")["sentiment_score"].mean()

## 8. Save merged, sentiment-enriched dataset to Google Drive

from google.colab import drive
drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/churn_prediction_project"
os.makedirs(SAVE_DIR, exist_ok=True)

csv_path = os.path.join(SAVE_DIR, "churn_with_sentiment.csv")
xlsx_path = os.path.join(SAVE_DIR, "churn_with_sentiment.xlsx")

final_df.to_csv(csv_path, index=False)
final_df.to_excel(xlsx_path, index=False, sheet_name="churn_with_sentiment")

print("Saved:", csv_path)
print("Saved:", xlsx_path)
final_df.shape

In [ ]:
final_df = (
    churn_df
    .drop_duplicates()                          # exact duplicate rows
    .drop_duplicates(subset="CustomerID")        # safety net, already unique from step 4
    .drop(columns=["dominant_tier"])             # internal helper, not real customer data
    .reset_index(drop=True)
)

print("Before cleanup:", churn_df.shape, "-> After cleanup:", final_df.shape)
print("Remaining nulls:", final_df.isna().sum().sum())
print("Remaining duplicate rows:", final_df.duplicated().sum())
final_df.head()

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/churn_prediction_project"
os.makedirs(SAVE_DIR, exist_ok=True)

out_path = os.path.join(SAVE_DIR, "churn_with_sentiment.csv")
churn_df.to_csv(out_path, index=False)
print("Saved:", out_path)
churn_df.shape

---
**Next:** open `02_model_training_shap_retention.ipynb`, which reads `churn_with_sentiment.csv` from your Drive, and continues the pipeline (`ML Churn Prediction -> SHAP Reasons -> AI Retention Message`).